# Проект 3. Прогноз оттока клиентов телеком-компании

Нужно по данным о клиенте предсказать, уйдёт он из компании (`Churn = 1`) или останется (`Churn = 0`).

Данные: Telco Customer Churn от IBM, 7043 клиента и 21 столбец. Файл `WA_Fn-UseC_-Telco-Customer-Churn.csv` скачан с Kaggle (https://www.kaggle.com/datasets/blastchar/telco-customer-churn) и должен лежать в одной папке с ноутбуком.

Перед обучением данные нужно подготовить. Столбцы «Yes/No» переводим в 1/0, тип контракта кодируем числами 0, 1, 2 (помесячный, на год, на два года). В `TotalCharges` у нескольких новых клиентов стоит пробел вместо числа, им ставится 0, потому что они ещё ничего не заплатили. Часть категориальных признаков (тип интернета, способ оплаты и т.д.) для простоты не используется.

Метод: логистическая регрессия из `GLM`. Модель выдаёт вероятность ухода клиента, и если она больше 0.5, считаем, что клиент уйдёт. Данные делятся 80/20, качество смотрится по accuracy, precision, recall, F1 и матрице ошибок.

In [ ]:
import Pkg
Pkg.add(["DataFrames", "CSV", "GLM", "StatsBase", "Random"])

In [1]:
using DataFrames, CSV, GLM, StatsBase, Random

# ─────────────────────────────────────────────
# ЗАГРУЗКА
# ─────────────────────────────────────────────
df = DataFrame(CSV.File("WA_Fn-UseC_-Telco-Customer-Churn.csv"))
println("Размер: ", size(df))
println("Столбцы: ", names(df))

Размер: (7043, 21)
Столбцы: ["customerID", "gender", "SeniorCitizen", "Partner", "Dependents", "tenure", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod", "MonthlyCharges", "TotalCharges", "Churn"]


In [ ]:
# ─────────────────────────────────────────────
# ПРЕДОБРАБОТКА
# ─────────────────────────────────────────────

select!(df, Not([:customerID]))

df.Churn = ifelse.(df.Churn .== "Yes", 1, 0)

df.TotalCharges = [something(tryparse(Float64, strip(string(x))), 0.0)
                   for x in df.TotalCharges]

df.gender         = ifelse.(df.gender .== "Male", 1, 0)
df.Partner        = ifelse.(df.Partner .== "Yes", 1, 0)
df.Dependents     = ifelse.(df.Dependents .== "Yes", 1, 0)
df.PhoneService   = ifelse.(df.PhoneService .== "Yes", 1, 0)
df.PaperlessBilling = ifelse.(df.PaperlessBilling .== "Yes", 1, 0)

df.Contract = Int.(replace(df.Contract,
    "Month-to-month" => 0,
    "One year"       => 1,
    "Two year"       => 2))

select!(df, [:SeniorCitizen, :tenure, :MonthlyCharges, :TotalCharges,
             :Contract, :gender, :Partner, :Dependents,
             :PhoneService, :PaperlessBilling, :Churn])

println("После предобработки: ", size(df))
println("Доля ушедших: ", round(mean(df.Churn) * 100, digits=2), "%")

После предобработки: (7043, 11)
Доля ушедших: 26.54%


In [ ]:
# ─────────────────────────────────────────────
# РАЗДЕЛЕНИЕ НА TRAIN/TEST
# ─────────────────────────────────────────────
Random.seed!(42)
n = nrow(df)
idx = shuffle(1:n)
n_train = floor(Int, 0.8 * n)          
train = df[idx[1:n_train], :]
test  = df[idx[n_train+1:end], :]

println("Train: ", nrow(train), " | Test: ", nrow(test))

Train: 5634 | Test: 1409


In [4]:
# ─────────────────────────────────────────────
# ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ
# ─────────────────────────────────────────────
fm = @formula(Churn ~ SeniorCitizen + tenure + MonthlyCharges +
                      TotalCharges + Contract + gender +
                      Partner + Dependents + PhoneService +
                      PaperlessBilling)

logit_model = glm(fm, train, Binomial(), LogitLink())
coeftable(logit_model)

|                  |        Coef. | Std. Error |      z | Pr(>\|z\|) |   Lower 95% |    Upper 95% |
|:-----------------|-------------:|-----------:|-------:|:-----------|:------------|-------------:|
| (Intercept)      | -0.765902    | 0.170053   |  -4.50 |     <1e-05 | -1.0992     | -0.432606 |
| SeniorCitizen    |  0.326101    | 0.0916347  |   3.56 |     0.0004 |  0.146501   |  0.505702 |
| tenure           | -0.0546454   | 0.00658824 |  -8.29 |     <1e-15 | -0.0675581  | -0.0417327 |
| MonthlyCharges   |  0.0258093   | 0.00219242 |  11.77 |     <1e-31 |  0.0215122  |  0.0301064 |
| TotalCharges     |  0.000227004 | 7.4101e-5  |   3.06 |     0.0022 |  8.17686e-5 |  0.000372239 |
| Contract         | -1.00316     | 0.0828562  | -12.11 |     <1e-33 | -1.16556    | -0.840768 |
| gender           | -0.0588036   | 0.0709261  |  -0.83 |     0.4071 | -0.197816   |  0.0802091 |
| Partner          |  0.0388542   | 0.0850932  |   0.46 |     0.6480 | -0.127925   |  0.205634 |
| Dependents       | -0.269471    | 0.0990415  |  -2.72 |     0.0065 | -0.463589   | -0.0753534 |
| PhoneService     | -0.997576    | 0.13338    |  -7.48 |     <1e-13 | -1.259      | -0.736156 |
| PaperlessBilling |  0.410155    | 0.0812126  |   5.05 |     <1e-06 |  0.250981   |  0.569329 |

In [5]:
# ─────────────────────────────────────────────
# ПРЕДСКАЗАНИЯ
# ─────────────────────────────────────────────
probs = predict(logit_model, test)
preds = probs .> 0.5;

In [6]:
# ─────────────────────────────────────────────
# МЕТРИКИ
# ─────────────────────────────────────────────
tp = sum((preds .== 1) .& (test.Churn .== 1))   # уйдёт, предсказали уйдёт
fp = sum((preds .== 1) .& (test.Churn .== 0))   # останется, предсказали уйдёт
fn = sum((preds .== 0) .& (test.Churn .== 1))   # уйдёт, предсказали останется
tn = sum((preds .== 0) .& (test.Churn .== 0))   # останется, предсказали останется

accuracy  = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall    = tp / (tp + fn)
f1        = 2 * precision * recall / (precision + recall)

println("Accuracy:  ", round(accuracy  * 100, digits=2), "%")
println("Precision: ", round(precision * 100, digits=2), "%")
println("Recall:    ", round(recall    * 100, digits=2), "%")
println("F1-score:  ", round(f1,        digits=4))
println()
println("Матрица ошибок:")
println("  TP = $tp    FP = $fp")
println("  FN = $fn    TN = $tn")

Accuracy:  80.62%
Precision: 65.63%
Recall:    56.68%
F1-score:  0.6083

Матрица ошибок:
  TP = 212    FP = 111
  FN = 162    TN = 924


## Вывод

Accuracy получилась 80.6%, но сама по себе эта цифра мало что говорит. Уходит около 26.5% клиентов, поэтому модель, которая всегда отвечает «останется», уже дала бы примерно 73.5%. Так что выигрыш от модели около 7 процентных пунктов.

Важнее, насколько хорошо модель находит именно уходящих клиентов. В тесте их 374, модель нашла 212 (recall 56.7%). Из всех, кого она отметила как уходящих, правы 65.6% (precision). F1 = 0.61.

По коэффициентам видно, что сильнее всего на уход влияет тип контракта: чем он длиннее, тем реже уходят. Также вероятность ухода снижается с длительностью обслуживания (`tenure`) и растёт с размером ежемесячного платежа. Пол и наличие партнёра почти ни на что не влияют (p-value 0.41 и 0.65).

Если бы задача была в том, чтобы найти как можно больше уходящих клиентов, стоило бы понизить порог 0.5. Recall вырос бы за счёт precision. Также можно добавить отброшенные категориальные признаки.